In [1]:
"""
Knee abnormality report labeler — small model, batched, multi-GPU.

Same architecture as the original Qwen script (prompt -> JSON -> parse), but:
  * Qwen2.5-1.5B-Instruct instead of 7B  (~3GB fp16, fits trivially on one T4)
  * fp16, no quantization  (nf4 is SLOWER than fp16 on Turing; T4 has no bf16)
  * batched generation      (the actual 15-25x speedup)
  * chat template applied   (Instruct models degrade badly without it)
  * one full model replica per GPU, not device_map="auto" layer sharding

Kaggle 2x T4.
"""

import json
import logging
import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import pydicom
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # step up to -3B- if parse quality is poor
MODEL_NAME = "/kaggle/input/datasets/soumabhamajumdar2548/qwen25-1b5-instruct"
COMP_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
BATCH_SIZE = 32          # drop to 16 if you see OOM with long reports
MAX_INPUT_TOKENS = 1536
MAX_NEW_TOKENS = 96      # a 12-key JSON is ~70 tokens; 256 was wasting decode steps
CHECKPOINT_EVERY = 5     # batches

LABEL_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

SYSTEM_PROMPT = (
    "You are a musculoskeletal radiologist extracting structured findings from "
    "knee MRI reports. You reply with a single JSON object and nothing else."
)


# --------------------------------------------------------------------------- #
# DICOM metadata (unchanged behaviour, trimmed fields)
# --------------------------------------------------------------------------- #

class DicomMeta:
    KEEP = ("Modality", "SeriesDescription", "BodyPartExamined", "Laterality")

    def __init__(self, base_path):
        self.base_path = base_path
        self.cache = {}

    def get(self, study_uid):
        if study_uid in self.cache:
            return self.cache[study_uid]

        study_path = Path(self.base_path) / str(study_uid)
        if not study_path.exists():
            self.cache[study_uid] = {}
            return {}

        dcm_files = list(study_path.rglob("*.dcm"))
        if not dcm_files:
            self.cache[study_uid] = {}
            return {}

        try:
            ds = pydicom.dcmread(dcm_files[0], stop_before_pixels=True)
            meta = {}
            for key in self.KEEP:
                val = ds.get(key, "")
                val = "" if val is None else str(val).strip()
                if val and val.upper() not in ("N/A", "NONE", "NULL"):
                    meta[key] = val
            self.cache[study_uid] = meta
            return meta
        except Exception as e:  # noqa: BLE001
            logging.warning(f"Error reading DICOM for {study_uid}: {e}")
            self.cache[study_uid] = {}
            return {}


# --------------------------------------------------------------------------- #
# One model replica pinned to one GPU
# --------------------------------------------------------------------------- #

class Replica:
    def __init__(self, model_name, device):
        self.device = device
        logging.info(f"Loading {model_name} onto {device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        ).to(device)
        self.model.eval()
        self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
        logging.info(f"Ready on {device}")

    @torch.inference_mode()
    def generate(self, prompts):
        texts = [
            self.tokenizer.apply_chat_template(
                [{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user", "content": p}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for p in prompts
        ]
        enc = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to(self.device)

        out = self.model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        gen = out[:, enc["input_ids"].shape[1]:]
        return self.tokenizer.batch_decode(gen, skip_special_tokens=True)


# --------------------------------------------------------------------------- #
# Labeler
# --------------------------------------------------------------------------- #

class KneeAbnormalityLabeler:
    def __init__(self, model_name=MODEL_NAME,
                 checkpoint_path="labeling_checkpoint.json",
                 dicom_base_path=f"{COMP_ROOT}/train_series/",
                 n_gpus=None):
        self.checkpoint_path = checkpoint_path
        self.dicom = DicomMeta(dicom_base_path)
        self.results = {}

        if os.path.exists(checkpoint_path):
            with open(checkpoint_path) as f:
                self.results = json.load(f)
            logging.info(f"Loaded checkpoint with {len(self.results)} completed rows")

        if n_gpus is None:
            n_gpus = max(1, torch.cuda.device_count())
        devices = [f"cuda:{i}" for i in range(n_gpus)] if torch.cuda.is_available() else ["cpu"]
        self.replicas = [Replica(model_name, d) for d in devices]
        self.pool = ThreadPoolExecutor(max_workers=len(self.replicas))
        logging.info(f"{len(self.replicas)} replica(s) live")

    # ---- prompt ---------------------------------------------------------- #

    def create_prompt(self, report_text, dicom_metadata=None):
        keys = ", ".join(f'"{c}"' for c in LABEL_COLUMNS)
        prompt = (
            "Read the knee MRI report below and decide, for each of 12 conditions, "
            "whether it is present.\n\n"
            f"Output a JSON object with exactly these keys: {keys}\n\n"
            "Values:\n"
            "  1  = condition is affirmatively described\n"
            "  0  = condition is explicitly negated or described as normal/intact\n"
            '  "?" = not mentioned, or the report is equivocal\n\n'
            "Do not infer a condition from an adjacent one. A meniscal tear does not "
            "imply an ACL tear. Respect laterality: medial findings go to medial keys "
            "only.\n\n"
        )
        if dicom_metadata:
            prompt += "Acquisition details:\n"
            for k, v in dicom_metadata.items():
                prompt += f"- {k}: {v}\n"
            prompt += "\n"
        prompt += f"REPORT:\n{report_text}\n\nJSON:"
        return prompt

    # ---- parsing (kept from original, slightly tightened) ---------------- #

    @staticmethod
    def parse_response(response_text):
        if not response_text:
            return None

        cleaned = re.sub(r"```(?:json)?\s*", "", response_text, flags=re.IGNORECASE)
        cleaned = re.sub(r"```\s*$", "", cleaned)

        start = cleaned.find("{")
        if start == -1:
            return None
        end = cleaned.rfind("}")

        if end != -1 and start < end:
            try:
                return json.loads(cleaned[start:end + 1])
            except json.JSONDecodeError:
                pass

        # Truncated output: close the object and drop any trailing partial pair.
        tail = cleaned[start:]
        tail = re.sub(r",\s*\"[^\"]*\"?\s*:?\s*[^,}]*$", "", tail)
        try:
            return json.loads(tail + "}")
        except json.JSONDecodeError:
            pass

        match = re.search(r"\{[^{}]*\}", cleaned)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return None

    # ---- batch dispatch --------------------------------------------------- #

    def _run_batch(self, prompts):
        """Split one batch across replicas, run in parallel, reassemble in order."""
        n = len(self.replicas)
        if n == 1:
            return self.replicas[0].generate(prompts)

        chunks = [prompts[i::n] for i in range(n)]
        futures = [
            self.pool.submit(rep.generate, chunk)
            for rep, chunk in zip(self.replicas, chunks) if chunk
        ]
        outs = [f.result() for f in futures]

        merged = [None] * len(prompts)
        for i, out in enumerate(outs):
            merged[i::n] = out
        return merged

    def save_checkpoint(self):
        with open(self.checkpoint_path, "w") as f:
            json.dump(self.results, f)
        logging.info(f"Checkpoint saved: {len(self.results)} rows")

    # ---- main loop -------------------------------------------------------- #

    def label_dataframe(self, df):
        result_df = df.copy()
        if "inferred" not in result_df.columns:
            result_df["inferred"] = ""

        todo = []
        for idx in range(len(df)):
            if str(idx) in self.results:
                continue
            row = df.iloc[idx]
            if not any(pd.isna(row[c]) for c in LABEL_COLUMNS):
                continue
            if pd.isna(row["Report"]) or not str(row["Report"]).strip():
                self.results[str(idx)] = {"error": "empty_report"}
                continue
            todo.append(idx)

        logging.info(f"{len(todo)} rows need labeling")

        n_batches = (len(todo) + BATCH_SIZE - 1) // BATCH_SIZE
        parse_failures = 0

        for b in tqdm(range(n_batches), desc="Labeling"):
            batch_idx = todo[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            prompts = []
            for idx in batch_idx:
                row = df.iloc[idx]
                meta = self.dicom.get(row["StudyInstanceUID"])
                prompts.append(self.create_prompt(str(row["Report"]), meta))

            responses = self._run_batch(prompts)

            for idx, resp in zip(batch_idx, responses):
                labels = self.parse_response(resp)
                if labels is None:
                    parse_failures += 1
                    self.results[str(idx)] = {"error": "parse_failed", "raw": resp[:200]}
                    continue

                inferred_cols = []
                for col in LABEL_COLUMNS:
                    if col not in labels:
                        continue
                    val = labels[col]
                    if val == "?" or val is None:
                        continue
                    try:
                        num = float(val)
                    except (ValueError, TypeError):
                        continue
                    if num in (0.0, 1.0):
                        result_df.at[idx, col] = int(num)
                        inferred_cols.append(col)

                if inferred_cols:
                    result_df.at[idx, "inferred"] = ",".join(inferred_cols)
                self.results[str(idx)] = {"completed": True, "inferred": inferred_cols}

            if (b + 1) % CHECKPOINT_EVERY == 0:
                self.save_checkpoint()

        self.save_checkpoint()
        logging.info(f"Parse failures: {parse_failures} / {len(todo)}")
        return result_df


def main():
    df = pd.read_csv(f"{COMP_ROOT}/train.csv")
    logging.info(f"Loaded {len(df)} rows; columns: {df.columns.tolist()}")

    labeler = KneeAbnormalityLabeler()
    labeled_df = labeler.label_dataframe(df)
    labeled_df.to_csv("train_labeled.csv", index=False)

    print("\n" + "=" * 50)
    print("LABELING SUMMARY")
    print("=" * 50)
    print(f"Total rows: {len(labeled_df)}")
    print(f"Rows with inferred labels: {labeled_df['inferred'].str.len().gt(0).sum()}")
    print("\nLabel distribution:")
    for col in LABEL_COLUMNS:
        non_nan = labeled_df[col].notna().sum()
        inferred = labeled_df["inferred"].str.contains(re.escape(col), na=False).sum()
        print(f"  {col:20s}: {non_nan:6d} total, {inferred:6d} inferred")


if __name__ == "__main__":
    main()

2026-09-12 21:40:45,794 - INFO - Loaded 4407 rows; columns: ['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
2026-09-12 21:40:46,107 - INFO - Loading /kaggle/input/datasets/soumabhamajumdar2548/qwen25-1b5-instruct onto cuda:0...
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2026-09-12 21:40:54,094 - INFO - Ready on cuda:0
2026-09-12 21:40:54,095 - INFO - Loading /kaggle/input/datasets/soumabhamajumdar2548/qwen25-1b5-instruct onto cuda:1...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2026-09-12 21:40:57,605 - INFO - Ready on cuda:1
2026-09-12 21:40:57,606 - INFO - 2 replica(s) live
2026-09-12 21:40:57,796 - INFO - 4349 rows need labeling


Labeling:   0%|          | 0/136 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-09-12 21:41:56,641 - INFO - Checkpoint saved: 160 rows
2026-09-12 21:43:10,219 - INFO - Checkpoint saved: 320 rows
2026-09-12 21:44:16,641 - INFO - Checkpoint saved: 480 rows
2026-09-12 21:45:24,411 - INFO - Checkpoint saved: 640 rows
2026-09-12 21:46:41,899 - INFO - Checkpoint saved: 800 rows
2026-09-12 21:47:48,955 - INFO - Checkpoint saved: 960 rows
2026-09-12 21:48:58,337 - INFO - Checkpoint saved: 1120 rows
2026-09-12 21:50:08,646 - INFO - Checkpoint saved: 1280 rows
2026-09-12 21:51:22,640 - INFO - Checkpoint saved: 1440 rows
2026-09-12 21:52:39,446 - INFO - Checkpoint saved: 1600 rows
2026-09-12 21:53:51,035 - INFO - Checkpoint saved: 1760 rows
2026-09-12 21:54:59,790 - INFO 


LABELING SUMMARY
Total rows: 4407
Rows with inferred labels: 3995

Label distribution:
  ACL                 :   2775 total,   2717 inferred
  MCL                 :   2795 total,   2737 inferred
  Medial Meniscus     :   3682 total,   3624 inferred
  Lateral Meniscus    :   3507 total,   3449 inferred
  Medial OA           :   2904 total,   2846 inferred
  Lateral OA          :   2898 total,   2840 inferred
  PF OA               :   2778 total,   2720 inferred
  Effusion            :   3671 total,   3613 inferred
  Synovitis           :   2627 total,   2569 inferred
  Baker's             :   3000 total,   2942 inferred
  Contusion           :    180 total,    122 inferred
  Fracture            :     58 total,      0 inferred


In [2]:
"""
Phase A, step 1 — build the slice manifest.

Walks the competition DICOM tree and produces one row per slice. No pixels are
read here; this reads DICOM headers only, and everything downstream in the
preprocessing pipeline reads its inputs from the table this produces.

What each group of columns is for:

  identity     study_uid, series_uid, path
               how we group slices into series and series into studies.

  geometry     ipp_*, iop_*, depth, plane
               ipp / iop are the raw DICOM fields. `depth` is how far along the
               stacking direction each slice sits, and is what step 3 sorts by.
               `plane` (sagittal / coronal / axial) is computed from the
               geometry rather than parsed out of the free-text series
               description, which differs between hospitals.

  acquisition  series_description, laterality, series_number, instance_number
               `laterality` drives the left-knee flip in step 6.

  decoding     rescale_slope, rescale_intercept, photometric_interpretation
               needed by step 4. Grabbed now so step 4 never has to reopen
               headers.

  shape        rows, cols, pixel_spacing_*, slice_thickness, spacing_between
               used for sanity checks and for the resize decision in step 7.

Usage
-----
    from build_manifest import build_manifest, summarise_manifest

    df = build_manifest("train", limit=20)     # smoke test on 20 studies
    summarise_manifest(df)

    df = build_manifest("train")               # the real run
    df.to_parquet("/kaggle/working/manifest_train.parquet")
"""

from __future__ import annotations

import os
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from tqdm.auto import tqdm

COMP_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
OUTPUT_DIR = Path("/kaggle/working")

# Listing the fields explicitly lets pydicom skip the rest of the header. That
# matters when you are opening a few hundred thousand files.
DICOM_TAGS = [
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SeriesNumber",
    "InstanceNumber",
    "ImagePositionPatient",
    "ImageOrientationPatient",
    "SeriesDescription",
    "Laterality",
    "BodyPartExamined",
    "Modality",
    "Rows",
    "Columns",
    "PixelSpacing",
    "SliceThickness",
    "SpacingBetweenSlices",
    "PhotometricInterpretation",
    "RescaleSlope",
    "RescaleIntercept",
]

PLANES = ("sagittal", "coronal", "axial")


# --------------------------------------------------------------------------- #
# Geometry
#
# ImageOrientationPatient is six numbers: the first three say which way you
# travel through the patient as you step one pixel RIGHT in the image, the last
# three as you step one pixel DOWN. ImagePositionPatient is where the slice's
# top-left corner sits, in millimetres.
# --------------------------------------------------------------------------- #

def slice_normal(iop):
    """The direction the slices stack along — perpendicular to the image."""
    row_dir = np.asarray(iop[:3], dtype=float)
    col_dir = np.asarray(iop[3:], dtype=float)
    return np.cross(row_dir, col_dir)


def slice_depth(ipp, normal):
    """How far along the stacking direction this slice sits, in millimetres.

    One number per slice. Sorting a series by this puts it in anatomical
    order, which sorting by filename or InstanceNumber does not reliably do.
    """
    return float(np.dot(np.asarray(ipp, dtype=float), normal))


def plane_from_normal(normal):
    """sagittal / coronal / axial, from which patient axis the stack runs along.

    Patient axes are x = toward the left, y = toward the back, z = toward the
    head. Slices stacking along x are side views (sagittal), along y are front
    views (coronal), along z are top-down (axial).
    """
    return PLANES[int(np.argmax(np.abs(normal)))]


# --------------------------------------------------------------------------- #
# Reading one slice's header
# --------------------------------------------------------------------------- #

def _get(ds, name, default=None):
    value = getattr(ds, name, None)
    return default if value is None else value


def _floats(value):
    """DICOM multi-valued numbers come back as a MultiValue, not a list."""
    return None if value is None else [float(v) for v in value]


def read_slice_header(path: Path, study_dir_name: str) -> dict:
    ds = pydicom.dcmread(str(path), stop_before_pixels=True, specific_tags=DICOM_TAGS)

    ipp = _floats(_get(ds, "ImagePositionPatient"))
    iop = _floats(_get(ds, "ImageOrientationPatient"))
    spacing = _floats(_get(ds, "PixelSpacing"))

    row = {
        "study_uid": str(_get(ds, "StudyInstanceUID", "")),
        "study_dir": study_dir_name,
        "series_uid": str(_get(ds, "SeriesInstanceUID", "")),
        "path": str(path),

        "series_number": _get(ds, "SeriesNumber"),
        "instance_number": _get(ds, "InstanceNumber"),
        "series_description": str(_get(ds, "SeriesDescription", "")),
        "laterality": str(_get(ds, "Laterality", "")),
        "body_part": str(_get(ds, "BodyPartExamined", "")),
        "modality": str(_get(ds, "Modality", "")),

        "rows": _get(ds, "Rows"),
        "cols": _get(ds, "Columns"),
        "pixel_spacing_row": spacing[0] if spacing else None,
        "pixel_spacing_col": spacing[1] if spacing else None,
        "slice_thickness": _get(ds, "SliceThickness"),
        "spacing_between_slices": _get(ds, "SpacingBetweenSlices"),

        # Everything step 4 needs, so it never has to reopen the header.
        "photometric_interpretation": str(_get(ds, "PhotometricInterpretation", "")),
        "rescale_slope": _get(ds, "RescaleSlope"),
        "rescale_intercept": _get(ds, "RescaleIntercept"),

        "error": None,
    }

    # Raw geometry, kept so we never have to rescan.
    for i, key in enumerate(("ipp_x", "ipp_y", "ipp_z")):
        row[key] = ipp[i] if ipp else None
    for i, key in enumerate(("iop_rx", "iop_ry", "iop_rz", "iop_cx", "iop_cy", "iop_cz")):
        row[key] = iop[i] if iop else None

    # Derived geometry. Both come from the two vectors above, so we compute
    # them here rather than carrying nine columns around and redoing it later.
    if iop is not None:
        normal = slice_normal(iop)
        row["plane"] = plane_from_normal(normal)
        row["depth"] = slice_depth(ipp, normal) if ipp is not None else None
    else:
        row["plane"] = None
        row["depth"] = None

    return row


def scan_study(study_dir: Path) -> list[dict]:
    """Every .dcm under one study folder. Errors are recorded, not raised —
    one unreadable file should not kill a 40-minute scan."""
    rows = []
    for path in sorted(Path(study_dir).rglob("*.dcm")):
        try:
            rows.append(read_slice_header(path, Path(study_dir).name))
        except Exception as exc:  # noqa: BLE001
            rows.append({
                "path": str(path),
                "study_dir": Path(study_dir).name,
                "error": repr(exc),
            })
    return rows


# --------------------------------------------------------------------------- #
# Driver
# --------------------------------------------------------------------------- #

def build_manifest(split: str = "train", limit: int | None = None,
                   workers: int | None = None) -> pd.DataFrame:
    """One row per slice for every study in {split}_series/.

    `limit` caps the number of studies — use it to smoke-test on 20 studies
    before committing to the full scan.
    """
    root = COMP_ROOT / f"{split}_series"
    if not root.exists():
        raise FileNotFoundError(f"{root} does not exist")

    studies = sorted(p for p in root.iterdir() if p.is_dir())
    if limit is not None:
        studies = studies[:limit]

    workers = workers or os.cpu_count() or 2

    rows: list[dict] = []
    with ProcessPoolExecutor(max_workers=workers) as pool:
        for study_rows in tqdm(pool.map(scan_study, studies, chunksize=4),
                               total=len(studies), desc=f"scanning {split}"):
            rows.extend(study_rows)

    df = pd.DataFrame(rows)
    df["split"] = split
    return df


# --------------------------------------------------------------------------- #
# Summary
#
# This is the point of running the scan first: it answers the questions we
# would otherwise be guessing at for the rest of the pipeline.
# --------------------------------------------------------------------------- #

def summarise_manifest(df: pd.DataFrame) -> None:
    ok = df[df["error"].isna()] if "error" in df else df
    failed = len(df) - len(ok)

    print(f"slices        {len(df):,}   ({failed:,} failed to read)")
    print(f"series        {ok['series_uid'].nunique():,}")
    print(f"studies       {ok['study_uid'].nunique():,}")

    mismatch = (ok["study_uid"] != ok["study_dir"]).sum()
    print(f"folder name disagrees with header StudyInstanceUID: {mismatch:,} slices")

    print("\nplane")
    print(ok["plane"].value_counts(dropna=False).to_string())

    print("\nlaterality")
    print(ok["laterality"].value_counts(dropna=False).to_string())

    per_series = ok.groupby("series_uid").size()
    print("\nslices per series")
    print(per_series.describe().to_string())
    print(f"series with < 8 slices (step 2 drops these): "
          f"{(per_series < 8).sum():,} of {len(per_series):,}")

    # Answers 'do I actually need to apply rescale slope/intercept?'
    print("\n(rescale_slope, rescale_intercept) combinations")
    pairs = ok[["rescale_slope", "rescale_intercept"]].astype(str)
    print(pairs.value_counts(dropna=False).head(10).to_string())

    # Answers 'do I need the MONOCHROME1 inversion?'
    print("\nphotometric interpretation")
    print(ok["photometric_interpretation"].value_counts(dropna=False).to_string())

    print("\nimage size")
    print(ok[["rows", "cols"]].astype(str).value_counts().head(10).to_string())

    print("\nmost common series descriptions")
    print(ok["series_description"].value_counts().head(20).to_string())


# if __name__ == "__main__":
#     OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#     for split in ("train", "test"):
#         manifest = build_manifest(split)
#         out = OUTPUT_DIR / f"manifest_{split}.parquet"
#         manifest.to_parquet(out, index=False)
#         print(f"\n=== {split} -> {out} ===")
#         summarise_manifest(manifest)

manifest_train = build_manifest("train")
manifest_train.to_parquet("/kaggle/working/manifest_train.parquet", index=False)
summarise_manifest(manifest_train)

manifest_test = build_manifest("test")
manifest_test.to_parquet("/kaggle/working/manifest_test.parquet", index=False)
summarise_manifest(manifest_test)


scanning train:   0%|          | 0/4407 [00:00<?, ?it/s]

slices        819,078   (0 failed to read)
series        24,371
studies       4,407
folder name disagrees with header StudyInstanceUID: 0 slices

plane
plane
sagittal    340843
coronal     243374
axial       234861

laterality
laterality
         364593
L        226483
R        225789
RIGHT      1327
LEFT        743
B           143

slices per series
count    24371.000000
mean        33.608715
std         26.575246
min         11.000000
25%         25.000000
50%         30.000000
75%         34.000000
max        320.000000
series with < 8 slices (step 2 drops these): 0 of 24,371

(rescale_slope, rescale_intercept) combinations
rescale_slope     rescale_intercept
nan               nan                  468607
1.0               0.0                  227583
1.5081807081807   0.0                     400
5.11037851037851  0.0                     320
4.18095238095238  0.0                     320
3.41001221001221  0.0                     320
4.44908424908425  0.0                     320
4.46007

scanning test:   0%|          | 0/3 [00:00<?, ?it/s]

slices        557   (0 failed to read)
series        15
studies       3
folder name disagrees with header StudyInstanceUID: 0 slices

plane
plane
axial       267
sagittal    180
coronal     110

laterality
laterality
L    284
     273

slices per series
count     15.000000
mean      37.133333
std       34.321727
min       24.000000
25%       24.000000
50%       30.000000
75%       31.500000
max      160.000000
series with < 8 slices (step 2 drops these): 0 of 15

(rescale_slope, rescale_intercept) combinations
rescale_slope  rescale_intercept
1.0            0.0                  284
nan            nan                  273

photometric interpretation
photometric_interpretation
MONOCHROME2    557

image size
rows  cols
512   512     195
640   1280    160
      640      54
960   960      34
640   512      30
800   800      30
896   896      30
672   672      24

most common series descriptions
series_description
t2_de3d_we_tra_Patella_fit_T    160
Ax PD FSE FS                     40
pd_tse

In [3]:
"""
Phase A, step 2 — filter and select the series we will actually cache.

Input is the slice manifest from step 1. Output is the exact list of slices the
cache builder should process. Nothing here reads pixels.

Four things happen, in order:

  laterality     normalise L/LEFT/R/RIGHT to one letter, then fill blanks from
                 other series in the same study. 44% of slices arrive with no
                 Laterality field, and step 6's left-knee flip depends on it.

  step 2   drop  slices we cannot use (unreadable, no geometry, duplicate
                 position) and series we do not want (localizers, non-MR,
                 too short, inconsistent size, mixed planes, bilateral).

  step 2b  select at most N series per plane per study. The dataset has ~5.5
                 series per study, which is more than the model needs and more
                 than fits on disk. We keep the most informative one per plane.

  step 6b  thin  each kept series to roughly one slice every few millimetres,
                 with a hard cap. 3D sequences arrive with 320 sub-millimetre
                 slices; thinning them to the same through-plane spacing as the
                 2D sequences makes every series comparable and makes the cache
                 fit.

Usage
-----
    from filter_series import filter_manifest, summarise_filter

    kept, report = filter_manifest(manifest)
    summarise_filter(manifest, kept, report)

    report[report.reject_reason.notna()].head(50)      # what was dropped
    report[~report.selected & report.reject_reason.isna()]  # kept but not chosen
"""

from __future__ import annotations

import re

import numpy as np
import pandas as pd

MIN_SLICES = 8
MAX_SERIES_PER_PLANE = 1        # step 2b: how many series to keep per plane
TARGET_SPACING_MM = 3.5         # step 6b: desired gap between kept slices
MAX_SLICES_PER_SERIES = 24      # step 6b: hard cap, drives the cache size

BYTES_PER_SLICE = 224 * 224     # uint8 at the cache resolution

# Long distinctive words can be matched anywhere in the description; short ones
# are matched as whole tokens.
LOCALIZER_SUBSTRINGS = ("localizer", "localiser", "scout", "survey", "topogram",
                        "scanogram", "tracker", "smartbrain", "calibration",
                        "3-plane", "3 plane")
LOCALIZER_TOKENS = {"loc", "plan", "ref", "cal"}

# Slice positions are floats; round before comparing so two slices genuinely at
# the same place compare equal. 0.01 mm is far below any real slice gap.
DEPTH_ROUNDING = 2


# --------------------------------------------------------------------------- #
# Reading the free-text series description
#
# Descriptions look like "pd_tse_fs_sag_320" or "SAG 3D_VIEW_PD_SPAIR_HR L".
# Splitting on every non-alphanumeric character turns those into clean tokens,
# which is safer than regex word boundaries — in regex an underscore counts as
# a word character, so \bfs\b does NOT match inside "pd_tse_fs_sag".
# --------------------------------------------------------------------------- #

def _tokens(description: str) -> set[str]:
    return {t for t in re.split(r"[^a-z0-9]+", (description or "").lower()) if t}


FAT_SAT_TOKENS = {"fs", "fatsat", "spair", "spir", "stir", "fsat", "spectral"}
PD_TOKENS = {"pd", "pdw", "proton"}
T2_TOKENS = {"t2", "t2w"}
T1_TOKENS = {"t1", "t1w"}
THREE_D_TOKENS = {"3d", "de3d", "dess", "space", "vibe", "wats", "fiesta", "medic"}


def classify_sequence(description: str) -> str:
    """Coarse sequence type, used to decide which series is worth keeping.

    Fat suppression is treated as a modifier rather than a type, so a 3D PD
    SPAIR still classifies as pd_fs — it is an excellent meniscus sequence and
    should not be downranked for being 3D.
    """
    t = _tokens(description)
    if not t:
        return "unknown"

    fat = bool(t & FAT_SAT_TOKENS)

    if "stir" in t:
        return "stir"
    if t & PD_TOKENS:
        return "pd_fs" if fat else "pd"
    if t & T2_TOKENS:
        return "t2_fs" if fat else "t2"
    if t & T1_TOKENS:
        return "t1"
    if t & THREE_D_TOKENS:
        return "3d"
    return "unknown"


# Higher is better. Fluid-sensitive fat-suppressed sequences show meniscal
# tears, effusion, synovitis, bone oedema and contusion — most of our labels.
# T1 is mainly for anatomy and marrow, so it ranks last. "unknown" sits mid
# table on purpose: 19% of this dataset has its description stripped, and we
# must not systematically exclude all of it.
SEQUENCE_SCORE = {
    "stir": 100,
    "pd_fs": 100,
    "t2_fs": 95,
    "pd": 70,
    "t2": 65,
    "unknown": 55,
    "3d": 50,
    "t1": 40,
}


def _looks_like_localizer(description: str) -> bool:
    text = (description or "").lower()
    if any(word in text for word in LOCALIZER_SUBSTRINGS):
        return True
    return bool(_tokens(description) & LOCALIZER_TOKENS)


# --------------------------------------------------------------------------- #
# Laterality
# --------------------------------------------------------------------------- #

def normalise_laterality(manifest: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """One letter per slice, filled in from the rest of the study where blank.

    Every series in a study images the same knee, so a study where any series
    recorded laterality can fill in all the others. Whatever is still blank
    after that needs the geometry fallback in step 6.
    """
    m = manifest.copy()
    raw = m["laterality"].fillna("").astype(str).str.strip().str.upper()

    # "LEFT" -> "L", "RIGHT" -> "R", "B" (bilateral) kept so step 2 can drop it.
    letter = raw.str[:1]
    m["laterality"] = letter.where(letter.isin(["L", "R", "B"]), "")

    before_blank = int((m["laterality"] == "").sum())

    # Propagate the study's known value into its blank slices.
    known = m.loc[m["laterality"].isin(["L", "R"])]
    by_study = known.groupby("study_uid")["laterality"].agg(
        lambda s: s.mode().iat[0] if not s.mode().empty else "")
    filled = m["study_uid"].map(by_study).fillna("")
    m["laterality"] = m["laterality"].where(m["laterality"] != "", filled)

    stats = {
        "blank_before": before_blank,
        "blank_after": int((m["laterality"] == "").sum()),
        "studies_with_none": int(
            m.groupby("study_uid")["laterality"]
             .agg(lambda s: not s.isin(["L", "R"]).any()).sum()),
    }
    return m, stats


# --------------------------------------------------------------------------- #
# Step 2 — slice-level cleaning
# --------------------------------------------------------------------------- #

def _drop_unusable_slices(manifest: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Remove slices we cannot place in space, and duplicate positions.

    Duplicate positions are the interesting case. Some series arrive with two
    slices at the same physical location — multi-echo sequences, or magnitude
    and phase images filed into one folder. Sorting those by depth produces a
    stack that alternates between two different images, which would poison the
    2.5D channel stack. We keep the first by instance number and flag it.
    """
    counts = {"start": len(manifest)}
    m = manifest

    if "error" in m.columns:
        m = m[m["error"].isna()]
    counts["read_errors"] = counts["start"] - len(m)

    before = len(m)
    m = m.dropna(subset=["depth", "plane", "series_uid"])
    counts["missing_geometry"] = before - len(m)

    before = len(m)
    m = m.assign(_depth_key=m["depth"].round(DEPTH_ROUNDING))
    m = (m.sort_values(["series_uid", "_depth_key", "instance_number"])
           .drop_duplicates(subset=["series_uid", "_depth_key"], keep="first")
           .drop(columns="_depth_key"))
    counts["duplicate_positions"] = before - len(m)

    return m, counts


# --------------------------------------------------------------------------- #
# Step 2 — series-level summary and rules
# --------------------------------------------------------------------------- #

def build_series_table(manifest: pd.DataFrame) -> pd.DataFrame:
    """One row per series, with the aggregates the rules and ranking need."""
    m = manifest.sort_values(["series_uid", "depth"]).copy()

    # A plain string is a version-proof way to count distinct image sizes.
    m["_size"] = m["rows"].astype(str) + "x" + m["cols"].astype(str)

    # Gap between consecutive slices. In a healthy series these are all about
    # the same — that is the slice spacing. Wildly uneven gaps mean the folder
    # holds more than one acquisition.
    m["_gap"] = m.groupby("series_uid")["depth"].diff()
    by_series = m.groupby("series_uid")
    gaps = by_series["_gap"]
    gap_mean, gap_std = gaps.mean(), gaps.std()

    table = pd.DataFrame({
        "study_uid": by_series["study_uid"].first(),
        "n_slices": by_series.size(),
        "plane": by_series["plane"].agg(lambda s: s.mode().iat[0]),
        "n_planes": by_series["plane"].nunique(),
        "series_description": by_series["series_description"].first(),
        "laterality": by_series["laterality"].first(),
        "modality": by_series["modality"].first(),
        "n_sizes": by_series["_size"].nunique(),
        "rows": by_series["rows"].first(),
        "cols": by_series["cols"].first(),
        "spacing_median": gaps.median().abs(),
        "spacing_cv": (gap_std / gap_mean.abs().replace(0, np.nan)).abs(),
    }).reset_index()

    table["sequence"] = table["series_description"].map(classify_sequence)
    table["sequence_score"] = table["sequence"].map(SEQUENCE_SCORE).fillna(50)
    return table


def _reject_reason(row, min_slices: int) -> str | None:
    """First rule that fires wins, so the report reads as one reason per series."""
    if str(row["modality"]).upper() not in ("MR", ""):
        return f"modality={row['modality']}"

    if _looks_like_localizer(row["series_description"]):
        return "localizer/scout"

    if row["n_slices"] < min_slices:
        return f"too few slices ({int(row['n_slices'])} < {min_slices})"

    # Both knees in frame — medial vs lateral is undefined, so the step 6 flip
    # cannot be applied and six of the twelve labels lose their meaning.
    if row["laterality"] == "B":
        return "bilateral"

    # Different image dimensions inside one series cannot be stacked into a
    # single volume in step 7.
    if row["n_sizes"] > 1:
        return "inconsistent image size"

    # Slices facing different directions in one series means two acquisitions
    # were filed together.
    if row["n_planes"] > 1:
        return "mixed planes in one series"

    return None


# --------------------------------------------------------------------------- #
# Step 2b — series selection
# --------------------------------------------------------------------------- #

def select_series(report: pd.DataFrame,
                  max_per_plane: int = MAX_SERIES_PER_PLANE) -> pd.DataFrame:
    """Mark the best `max_per_plane` surviving series in each study and plane.

    Ranking, in order:

      1. sequence score — fluid-sensitive fat-suppressed sequences first,
         because they carry most of our twelve findings.
      2. slice count closest to that plane's median. This is the tiebreak that
         does the work when the description is stripped, and it also pushes
         away both the unusually short series and the 320-slice 3D ones.
    """
    report = report.copy()
    report["selected"] = False

    alive = report["reject_reason"].isna()
    if not alive.any():
        return report

    plane_median = report.loc[alive].groupby("plane")["n_slices"].median()
    report["_slice_dist"] = (
        report["n_slices"] - report["plane"].map(plane_median)).abs()

    ranked = (report[alive]
              .sort_values(["study_uid", "plane", "sequence_score", "_slice_dist"],
                           ascending=[True, True, False, True]))
    ranked["_rank"] = ranked.groupby(["study_uid", "plane"]).cumcount()

    chosen = ranked.loc[ranked["_rank"] < max_per_plane, "series_uid"]
    report.loc[report["series_uid"].isin(set(chosen)), "selected"] = True
    return report.drop(columns="_slice_dist")


# --------------------------------------------------------------------------- #
# Step 6b — thin each series along the stack
# --------------------------------------------------------------------------- #

def _keep_positions(n: int, spacing_mm: float, target_spacing_mm: float,
                    max_slices: int) -> np.ndarray:
    """Which positions within a sorted series to keep.

    A 2D series already at ~3.5 mm spacing is untouched. A 3D series at 0.6 mm
    keeps every sixth slice, landing at comparable through-plane spacing — so
    the 2.5D neighbour stack spans a similar amount of anatomy on both.
    """
    if n <= 0:
        return np.array([], dtype=int)

    stride = 1
    if spacing_mm and spacing_mm > 0:
        stride = max(1, int(round(target_spacing_mm / spacing_mm)))

    idx = np.arange(0, n, stride)
    if len(idx) > max_slices:
        # Still too many (a long series covering a lot of leg): spread the cap
        # evenly across the whole stack so coverage is preserved.
        idx = np.unique(np.linspace(0, n - 1, max_slices).round().astype(int))
    return idx


def thin_series(manifest: pd.DataFrame, report: pd.DataFrame,
                target_spacing_mm: float = TARGET_SPACING_MM,
                max_slices: int = MAX_SLICES_PER_SERIES) -> pd.DataFrame:
    """Keep an evenly spaced subset of each series' slices."""
    spacing = report.set_index("series_uid")["spacing_median"].to_dict()

    m = manifest.sort_values(["series_uid", "depth"]).copy()
    m["_pos"] = m.groupby("series_uid").cumcount()

    keep = []
    for uid, group in m.groupby("series_uid", sort=False):
        positions = _keep_positions(len(group), spacing.get(uid, 0.0),
                                    target_spacing_mm, max_slices)
        keep.append(group[group["_pos"].isin(set(positions.tolist()))])

    return pd.concat(keep).drop(columns="_pos") if keep else m.drop(columns="_pos")


# --------------------------------------------------------------------------- #
# Driver
# --------------------------------------------------------------------------- #

def filter_manifest(manifest: pd.DataFrame,
                    min_slices: int = MIN_SLICES,
                    max_per_plane: int = MAX_SERIES_PER_PLANE,
                    target_spacing_mm: float = TARGET_SPACING_MM,
                    max_slices: int = MAX_SLICES_PER_SERIES,
                    ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returns (slices to cache, per-series report).

    The report has a row for every series seen. `reject_reason` says why a
    series was dropped by step 2; `selected` says whether step 2b chose it. A
    series can be perfectly usable and still not selected — that is the normal
    case, since most studies have more series than we need.
    """
    m, laterality_stats = normalise_laterality(manifest)
    m, slice_counts = _drop_unusable_slices(m)

    report = build_series_table(m)
    report["reject_reason"] = report.apply(_reject_reason, axis=1,
                                           min_slices=min_slices)
    report = select_series(report, max_per_plane=max_per_plane)

    chosen = set(report.loc[report["selected"], "series_uid"])
    kept = m[m["series_uid"].isin(chosen)].copy()
    slice_counts["after_selection"] = len(kept)

    kept = thin_series(kept, report, target_spacing_mm, max_slices)
    slice_counts["after_thinning"] = len(kept)

    kept.attrs["slice_counts"] = slice_counts
    kept.attrs["laterality_stats"] = laterality_stats
    return kept, report


# --------------------------------------------------------------------------- #
# Summary
# --------------------------------------------------------------------------- #

def summarise_filter(original: pd.DataFrame, kept: pd.DataFrame,
                     report: pd.DataFrame) -> None:
    counts = kept.attrs.get("slice_counts", {})
    lat = kept.attrs.get("laterality_stats", {})

    print("laterality")
    print(f"  blank before study-level fill  {lat.get('blank_before', 0):,}")
    print(f"  blank after                    {lat.get('blank_after', 0):,}")
    print(f"  studies with no laterality at all  {lat.get('studies_with_none', 0):,}"
          "   <- these need the geometry fallback in step 6")

    print("\nslice-level removals")
    for name in ("read_errors", "missing_geometry", "duplicate_positions"):
        print(f"  {name:22} {counts.get(name, 0):,}")

    print("\nseries-level removals (step 2)")
    print(report["reject_reason"].fillna("KEPT").value_counts().to_string())

    alive = report[report["reject_reason"].isna()]
    print(f"\nseries selection (step 2b)")
    print(f"  usable series      {len(alive):,}")
    print(f"  selected           {int(report['selected'].sum()):,}")
    print("  selected by sequence type:")
    print(report[report["selected"]]["sequence"].value_counts()
          .to_string().replace("\n", "\n    "))

    print("\nslices")
    print(f"  manifest                {len(original):,}")
    print(f"  after step 2 + 2b       {counts.get('after_selection', 0):,}")
    print(f"  after thinning (6b)     {counts.get('after_thinning', 0):,}")

    gb = len(kept) * BYTES_PER_SLICE / 1e9
    print(f"\nprojected cache at 224x224 uint8: {gb:.1f} GB"
          f"   {'OK' if gb < 19 else 'TOO BIG -- lower MAX_SLICES_PER_SERIES'}")

    print(f"\nstudies   {original['study_uid'].nunique():,} -> "
          f"{kept['study_uid'].nunique():,}")
    lost = set(original["study_uid"].dropna()) - set(kept["study_uid"])
    print(f"studies with nothing left: {len(lost):,}")
    if lost:
        print("  " + ", ".join(sorted(lost)[:5]) + (" ..." if len(lost) > 5 else ""))

    print("\nselected series per study")
    per_study = report[report["selected"]].groupby("study_uid").size()
    print(per_study.value_counts().sort_index().to_string())

    print("\nplanes present among selected series")
    print(report[report["selected"]]["plane"].value_counts().to_string())

    print("\ndescriptions dropped as localizer/scout (check these are junk)")
    dropped = report[report["reject_reason"] == "localizer/scout"]
    print(dropped["series_description"].value_counts().head(15).to_string()
          if len(dropped) else "  none")

    uneven = alive[alive["spacing_cv"] > 0.1]
    print(f"\nusable series with uneven slice spacing (cv > 0.1): {len(uneven):,}")

In [4]:
# --- load the manifest -------------------------------------------------
# still in memory from the scan:
manifest_train = manifest_train
# or after a kernel restart:
# manifest_train = pd.read_parquet("/kaggle/working/manifest_train.parquet")

# --- run step 2 + 2b + 6b ----------------------------------------------
kept_train, report_train = filter_manifest(manifest_train)
summarise_filter(manifest_train, kept_train, report_train)

laterality
  blank before study-level fill  364,593
  blank after                    357,849
  studies with no laterality at all  2,204   <- these need the geometry fallback in step 6

slice-level removals
  read_errors            0
  missing_geometry       0
  duplicate_positions    0

series-level removals (step 2)
reject_reason
KEPT         24367
bilateral        4

series selection (step 2b)
  usable series      24,367
  selected           13,218
  selected by sequence type:
sequence
    pd_fs      6929
    unknown    3370
    t2_fs      1798
    pd          699
    stir        345
    t2           70
    t1            7

slices
  manifest                819,078
  after step 2 + 2b       436,177
  after thinning (6b)     308,590

projected cache at 224x224 uint8: 15.5 GB   OK

studies   4,407 -> 4,406
studies with nothing left: 1
  1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588

selected series per study
3    4406

planes present among selected series
plane
axial 

In [5]:
kept_train.to_parquet("/kaggle/working/cache_index_train.parquet", index=False)
report_train.to_parquet("/kaggle/working/series_report_train.parquet", index=False)